# Lab 4: ML Model Training and Serving with FastAPI

## Goal

- Train a fraud detection model offline on historical data,
- Serve the model via FastAPI,
- Connect the API to a Kafka consumer for real-time scoring.

**Business case (continued):** Your rule-based system works, but you want to improve accuracy with a machine learning model.

---

## Part 1: Train the Model

### Task 1.1 -- Generate training data

Create a dataset that mimics our Kafka transactions -- with known fraud labels.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import pickle

np.random.seed(42)
n_normal = 2000
n_fraud = 100

normal = pd.DataFrame({
    'amount': np.random.lognormal(5, 1, n_normal).clip(5, 5000),
    'hour': np.random.normal(14, 4, n_normal).clip(0, 23).astype(int),
    'is_electronics': np.random.binomial(1, 0.3, n_normal),
    'tx_per_day': np.random.poisson(3, n_normal),
    'fraud': 0
})

fraud = pd.DataFrame({
    'amount': np.random.uniform(2000, 9000, n_fraud),
    'hour': np.random.choice([0,1,2,3,4,5,22,23], n_fraud),
    'is_electronics': np.random.binomial(1, 0.7, n_fraud),
    'tx_per_day': np.random.poisson(8, n_fraud),
    'fraud': 1
})

df = pd.concat([normal, fraud], ignore_index=True).sample(frac=1, random_state=42)
print(f"Dataset: {len(df)} rows, fraud rate: {df['fraud'].mean():.1%}")
df.head()

### Task 1.2 -- Train and evaluate a model

In [ ]:
features = ['amount', 'hour', 'is_electronics', 'tx_per_day']
X = df[features]
y = df['fraud']

# YOUR CODE
# 1. Split into train/test (80/20, stratify=y)
# 2. Train RandomForestClassifier(n_estimators=100)
# 3. Print classification_report on test set
# 4. Save model to 'fraud_model.pkl' with pickle

---

## Part 2: Serve the Model with FastAPI

### Task 2.1 -- Create the API

Write a FastAPI app that loads the model and exposes a `POST /score` endpoint.

In [ ]:
%%file fraud_api.py
from fastapi import FastAPI
from pydantic import BaseModel
import pickle
import numpy as np

app = FastAPI(title="Fraud Detection API")
model = pickle.load(open('fraud_model.pkl', 'rb'))

class Transaction(BaseModel):
    amount: float
    hour: int
    is_electronics: int
    tx_per_day: int

# YOUR CODE
# POST /score endpoint:
# - Accept Transaction object
# - Extract features as numpy array
# - model.predict() and model.predict_proba()
# - Return: {"is_fraud": bool, "fraud_probability": float, "features": dict}

Run the API in a **JupyterLab terminal**:
```bash
uvicorn fraud_api:app --host 0.0.0.0 --port 8001
```

### Task 2.2 -- Test the API

In [ ]:
import requests

# Test with a normal transaction
normal_tx = {"amount": 150, "hour": 14, "is_electronics": 0, "tx_per_day": 3}
response = requests.post("http://localhost:8001/score", json=normal_tx)
print("Normal:", response.json())

# Test with a suspicious transaction
# YOUR CODE

---

## Part 3: Connect API to Kafka

### Task 3.1 -- Scoring consumer with API calls

Write a Kafka consumer that sends each transaction to your FastAPI for scoring.

In [ ]:
%%file ml_consumer.py
from kafka import KafkaConsumer, KafkaProducer
import json, requests
from datetime import datetime

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='ml-scoring',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

alert_producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

API_URL = "http://localhost:8001/score"

# YOUR CODE
# For each transaction:
# 1. Extract features (amount, hour from timestamp, is_electronics, tx_per_day=estimate)
# 2. Call API: requests.post(API_URL, json=features)
# 3. If is_fraud: send alert to 'alerts' topic
# 4. Print summary every 10 messages

---

## Homework

1. Add a `/health` endpoint to the API that returns model metadata (accuracy, training date).
2. Create a Dockerfile for the API and add it to docker-compose.
3. Compare rule-based (Lab 3) vs ML-based detection: which catches more frauds?
4. Push to Git.

**Next lab:** Online/incremental learning -- a model that updates itself with new data.